In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import arff

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import json


In [ ]:

data, meta = arff.loadarff('data/electricity-normalized.arff')
df = pd.DataFrame(data)

# convertim bytes -> string (ARFF face asta uneori)
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].apply(lambda x: x.decode('utf-8') if isinstance(x, (bytes, bytearray)) else x)

df.head()


In [ ]:

X = df.drop(columns=['class'])
y = df['class']

# y e UP/DOWN (string), il facem 0/1
le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape


In [ ]:

pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000))
])

params_lr = {
    'clf__C': [0.1, 1, 10],
    'clf__penalty': ['l2'],
    'clf__solver': ['lbfgs']
}

gs_lr = GridSearchCV(pipe_lr, params_lr, cv=5, scoring='f1', n_jobs=-1)
gs_lr.fit(X_train, y_train)

gs_lr.best_params_, gs_lr.best_score_


In [ ]:

dt = DecisionTreeClassifier(random_state=42)

params_dt = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 10, 20],
    'min_samples_leaf': [1, 5, 10]
}

gs_dt = GridSearchCV(dt, params_dt, cv=5, scoring='f1', n_jobs=-1)
gs_dt.fit(X_train, y_train)

gs_dt.best_params_, gs_dt.best_score_


In [ ]:

rf = RandomForestClassifier(random_state=42)

params_rf = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 5]
}

gs_rf = GridSearchCV(rf, params_rf, cv=5, scoring='f1', n_jobs=-1)
gs_rf.fit(X_train, y_train)

gs_rf.best_params_, gs_rf.best_score_


In [ ]:

def eval_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    # roc_auc doar daca avem predict_proba
    auc = None
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, proba)

    return {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": auc
    }

best_lr = gs_lr.best_estimator_
best_dt = gs_dt.best_estimator_
best_rf = gs_rf.best_estimator_

results = []
results.append(eval_model("LogisticRegression", best_lr, X_test, y_test))
results.append(eval_model("DecisionTree", best_dt, X_test, y_test))
results.append(eval_model("RandomForest", best_rf, X_test, y_test))

results_df = pd.DataFrame(results)
results_df


In [ ]:

results_df.to_csv("results_metrics.csv", index=False)

best_params = {
    "LogisticRegression": gs_lr.best_params_,
    "DecisionTree": gs_dt.best_params_,
    "RandomForest": gs_rf.best_params_
}

with open("best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)

print("Salvat: results_metrics.csv si best_params.json")
